# Day 11 — Regression metrics: MAE, MSE, RMSE and R²

Run each cell with **Shift + Enter**, from the top. We use the 150-student dataset from Days 5-10, `student_data_150.csv`: 8 real students and 142 **synthetic** ones (simulated for practice, marked in the Source column).

## 1. Understanding regression metrics

In [ ]:
# Get the dataset: Colab starts with an empty folder, so download the file the first time
import os
import urllib.request

URL = "https://raw.githubusercontent.com/neerajb1/learnforge-notebooks/main/ai-ml-120/day-011/data/student_data_150.csv"
if not os.path.exists("student_data_150.csv"):
    urllib.request.urlretrieve(URL, "student_data_150.csv")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Three students with illustrative predictions
y_true = [80, 60, 90]
y_pred = [75, 65, 85]

mae = mean_absolute_error(y_true, y_pred)
mse = mean_squared_error(y_true, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_true, y_pred)
print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R2:", round(r2, 3))

MAE: 5.0
MSE: 25.0
RMSE: 5.0
R2: 0.839


In [ ]:
# R2 by hand: compare the model's squared error with the mean baseline's
y, p = np.array(y_true), np.array(y_pred)
sse = ((y - p) ** 2).sum()              # the model's squared errors
sst = ((y - y.mean()) ** 2).sum()       # squared errors of always predicting the mean of these actual values
print(f"Mean of the actual values: {y.mean():.2f}")
print(f"Model squared error: {sse:.1f} | mean baseline squared error: {sst:.1f}")
print(f"R2 = 1 - {sse:.1f} / {sst:.1f} = {1 - sse / sst:.3f}")

Mean of the actual values: 76.67
Model squared error: 75.0 | mean baseline squared error: 466.7
R2 = 1 - 75.0 / 466.7 = 0.839


In [ ]:
# Two models, same students: several small errors vs mostly small with one large
actual = np.array([55, 62, 48, 70, 66, 59, 73, 51, 64, 68])
model_a = actual + np.array([4, -4, 4, -4, 4, -4, 4, -4, 4, -4])     # every prediction 4 marks off
model_b = actual + np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 40])         # perfect, except one 40 marks off

for name, pred in [("Model A", model_a), ("Model B", model_b)]:
    mse = mean_squared_error(actual, pred)
    print(f"{name}: MAE {mean_absolute_error(actual, pred):.2f} | MSE {mse:.2f} | RMSE {np.sqrt(mse):.2f}"
          f" | R2 {r2_score(actual, pred):.3f}")

Model A: MAE 4.00 | MSE 16.00 | RMSE 4.00 | R2 0.740
Model B: MAE 4.00 | MSE 160.00 | RMSE 12.65 | R2 -1.604


In [ ]:
# How each metric reacts as ONE error grows (nine other students each 2 marks off)
rows = []
for big in range(0, 26, 5):
    errors = np.array([2] * 9 + [big])
    rows.append({"LargeError": big, "MAE": errors.mean(), "MSE": (errors ** 2).mean(),
                 "RMSE": np.sqrt((errors ** 2).mean())})
behaviour = pd.DataFrame(rows).round(2)
print(behaviour.to_string(index=False))
behaviour.to_csv("metric_behaviour.csv", index=False)

 LargeError  MAE  MSE  RMSE
          0  1.8  3.6  1.90
          5  2.3  6.1  2.47
         10  2.8 13.6  3.69
         15  3.3 26.1  5.11
         20  3.8 43.6  6.60
         25  4.3 66.1  8.13


In [ ]:
# R2 can be negative, and it isn't the share of correct predictions
for name, pred in [("Always predicts 70", np.full(10, 70)),
                   ("Reversed trend", actual.mean() - (actual - actual.mean()))]:
    print(f"{name:<19} R2 {r2_score(actual, pred):7.3f} | MAE {mean_absolute_error(actual, pred):5.2f}")
within_5 = (np.abs(model_a - actual) <= 5).mean()
print(f"Model A: {within_5:.0%} of predictions within 5 marks, but R2 is {r2_score(actual, model_a):.3f}")

Always predicts 70  R2  -1.148 | MAE  9.00
Reversed trend      R2  -3.000 | MAE 13.36
Model A: 100% of predictions within 5 marks, but R2 is 0.740


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# On our data: Day 10's split and chosen model, all four metrics on the VALIDATION students
df = pd.read_csv("student_data_150.csv")
train_val, test = train_test_split(df, test_size=0.15, random_state=42)
train, validation = train_test_split(train_val, test_size=0.15 / 0.85, random_state=42)
features = ["StudyHours", "Attendance", "Age"]
model = LinearRegression().fit(train[features], train["ExamScore"])
y_val, p_val = validation["ExamScore"], model.predict(validation[features])
mse = mean_squared_error(y_val, p_val)
print(f"MAE {mean_absolute_error(y_val, p_val):.2f} | MSE {mse:.2f} | RMSE {np.sqrt(mse):.2f} | R2 {r2_score(y_val, p_val):.3f}")
print(f"Exam scores: standard deviation {y_val.std(ddof=0):.2f} on validation, range {df['ExamScore'].min():.0f}-{df['ExamScore'].max():.0f}")

MAE 8.83 | MSE 110.24 | RMSE 10.50 | R2 0.312
Exam scores: standard deviation 12.66 on validation, range 24-91


In [ ]:
# Which mean does R2 compare with? The mean of the EVALUATION targets
val_mean = np.full(len(y_val), y_val.mean())
train_mean = np.full(len(y_val), train["ExamScore"].mean())
print(f"Predict the validation mean ({y_val.mean():.2f}): R2 {r2_score(y_val, val_mean):.3f}")
print(f"Predict the training mean ({train['ExamScore'].mean():.2f}):   R2 {r2_score(y_val, train_mean):.3f}")

Predict the validation mean (58.48): R2 0.000
Predict the training mean (57.08):   R2 -0.012


In [ ]:
# A simple view of R2: study hours alone, fitted on the training students
one = LinearRegression().fit(train[["StudyHours"]], train["ExamScore"])
print(f"Training R2 {r2_score(train['ExamScore'], one.predict(train[['StudyHours']])):.3f}"
      f" | validation R2 {r2_score(y_val, one.predict(validation[['StudyHours']])):.3f}")
train[["StudyHours", "ExamScore"]].to_csv("training_hours.csv", index=False)

Training R2 0.368 | validation R2 0.236


## 2. Interpreting R² in detail

In [ ]:
# Section 2: the R2 formula, one student at a time (the three students from Section 1)
mean_y = y.mean()
parts = pd.DataFrame({"Actual": y, "Predicted": p, "Residual squared": (y - p) ** 2,
                      "Distance from mean squared": ((y - mean_y) ** 2).round(2)})
print(parts.to_string(index=False))
ss_res, ss_tot = parts["Residual squared"].sum(), ((y - mean_y) ** 2).sum()
print(f"Mean {mean_y:.2f} | SS_res {ss_res:.2f} | SS_tot {ss_tot:.2f} | R2 = 1 - {ss_res:.0f} / {ss_tot:.2f} = {1 - ss_res / ss_tot:.3f}")
print(f"r2_score gives the same: {r2_score(y, p):.3f}")

 Actual  Predicted  Residual squared  Distance from mean squared
     80         75                25                       11.11
     60         65                25                      277.78
     90         85                25                      177.78
Mean 76.67 | SS_res 75.00 | SS_tot 466.67 | R2 = 1 - 75 / 466.67 = 0.839
r2_score gives the same: 0.839


In [ ]:
# What different R2 values look like: three real models, fitted on the training students, on the SAME validation students
fits = {"All three features": features, "Study hours only": ["StudyHours"], "Age only": ["Age"]}
shown = pd.DataFrame({"Actual": y_val.values})
for name, cols in fits.items():
    pred = LinearRegression().fit(train[cols], train["ExamScore"]).predict(validation[cols])
    shown[name] = pred.round(1)
    mse = mean_squared_error(y_val, pred)
    print(f"{name:<19} validation R2 {r2_score(y_val, pred):6.3f} | RMSE {np.sqrt(mse):5.2f} | MAE {mean_absolute_error(y_val, pred):5.2f}")
shown.columns = ["Actual", "AllFeatures", "StudyHoursOnly", "AgeOnly"]
shown.to_csv("r2_fits.csv", index=False)

All three features  validation R2  0.312 | RMSE 10.50 | MAE  8.83
Study hours only    validation R2  0.236 | RMSE 11.07 | MAE  9.78
Age only            validation R2 -0.026 | RMSE 12.83 | MAE 10.72


In [ ]:
# Adding irrelevant features: columns of random numbers, unrelated to the score
rng = np.random.default_rng(0)
rows = []
for k in [0, 5, 10, 20, 40, 60]:
    noise_tr = rng.normal(size=(len(train), k)); noise_va = rng.normal(size=(len(validation), k))
    X_tr = np.hstack([train[features].values, noise_tr]); X_va = np.hstack([validation[features].values, noise_va])
    m = LinearRegression().fit(X_tr, train["ExamScore"])
    rows.append({"RandomFeatures": k, "TrainingR2": round(r2_score(train["ExamScore"], m.predict(X_tr)), 3),
                 "ValidationR2": round(r2_score(y_val, m.predict(X_va)), 3)})
noise = pd.DataFrame(rows)
print(f"{len(train)} training students, {len(validation)} validation students")
print(noise.to_string(index=False))
noise.to_csv("noise_features.csv", index=False)

104 training students, 23 validation students
 RandomFeatures  TrainingR2  ValidationR2
              0       0.434         0.312
              5       0.481         0.249
             10       0.482         0.215
             20       0.577         0.070
             40       0.641         0.046
             60       0.770        -0.760


In [ ]:
# R2 depends on how much the targets vary: the same model, on all validation students and on those scoring 50-65
model_all = LinearRegression().fit(train[features], train["ExamScore"])
narrow = validation[validation["ExamScore"].between(50, 65)]
for name, part in [("All validation students", validation), ("Scores 50-65 only", narrow)]:
    pred = model_all.predict(part[features])
    print(f"{name:<24} n={len(part):2d} | score std {part['ExamScore'].std(ddof=0):5.2f}"
          f" | MAE {mean_absolute_error(part['ExamScore'], pred):5.2f} | R2 {r2_score(part['ExamScore'], pred):6.3f}")

All validation students  n=23 | score std 12.66 | MAE  8.83 | R2  0.312
Scores 50-65 only        n= 8 | score std  3.72 | MAE  7.41 | R2 -5.436


## 3. Model comparison and trade-offs

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge, Lasso
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Section 3: four models, one split, the same validation students for all of them
def scores(actual, predicted):
    mse = mean_squared_error(actual, predicted)
    return {"MAE": mean_absolute_error(actual, predicted), "MSE": mse, "RMSE": np.sqrt(mse), "R2": r2_score(actual, predicted)}

# Ridge and Lasso penalize coefficient size, which depends on each feature's units: scale the features first
models = {
    "Mean baseline": DummyRegressor(strategy="mean"),
    "Linear regression": LinearRegression(),
    "Ridge (alpha=1)": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    "Lasso (alpha=0.01)": make_pipeline(StandardScaler(), Lasso(alpha=0.01)),
}
rows = []
for name, m in models.items():
    m.fit(train[features], train["ExamScore"])
    rows.append({"Model": name, **scores(y_val, m.predict(validation[features]))})
comparison = pd.DataFrame(rows).round(3)
print(comparison.to_string(index=False))
comparison.to_csv("model_comparison.csv", index=False)

             Model    MAE     MSE   RMSE     R2
     Mean baseline 10.592 162.300 12.740 -0.012
 Linear regression  8.830 110.243 10.500  0.312
   Ridge (alpha=1)  8.823 110.130 10.494  0.313
Lasso (alpha=0.01)  8.829 110.219 10.499  0.313


In [ ]:
# How strong should the penalty be? Try several alphas, judged on validation
for name, make in [("Ridge", Ridge), ("Lasso", Lasso)]:
    for alpha in [0.01, 0.1, 1, 10, 100]:
        m = make_pipeline(StandardScaler(), make(alpha=alpha)).fit(train[features], train["ExamScore"])
        coef = m[-1].coef_
        print(f"{name:<5} alpha {alpha:>6}: validation MAE {mean_absolute_error(y_val, m.predict(validation[features])):5.2f}"
              f" | coefficients (scaled) {np.round(coef, 2).tolist()}")

Ridge alpha   0.01: validation MAE  8.83 | coefficients (scaled) [7.56, 3.73, 0.52]
Ridge alpha    0.1: validation MAE  8.83 | coefficients (scaled) [7.55, 3.73, 0.52]
Ridge alpha      1: validation MAE  8.82 | coefficients (scaled) [7.49, 3.71, 0.52]
Ridge alpha     10: validation MAE  8.80 | coefficients (scaled) [6.94, 3.54, 0.53]
Ridge alpha    100: validation MAE  9.06 | coefficients (scaled) [4.03, 2.32, 0.44]
Lasso alpha   0.01: validation MAE  8.83 | coefficients (scaled) [7.55, 3.72, 0.51]
Lasso alpha    0.1: validation MAE  8.82 | coefficients (scaled) [7.49, 3.64, 0.42]
Lasso alpha      1: validation MAE  8.78 | coefficients (scaled) [6.83, 2.87, 0.0]
Lasso alpha     10: validation MAE 10.59 | coefficients (scaled) [0.0, 0.0, 0.0]
Lasso alpha    100: validation MAE 10.59 | coefficients (scaled) [0.0, 0.0, 0.0]


In [ ]:
# Overfitting check: training error against validation error for each model
for name, m in models.items():
    tr = mean_absolute_error(train["ExamScore"], m.predict(train[features]))
    va = mean_absolute_error(y_val, m.predict(validation[features]))
    print(f"{name:<19} training MAE {tr:5.2f} | validation MAE {va:5.2f} | gap {va - tr:+.2f}")

Mean baseline       training MAE 10.99 | validation MAE 10.59 | gap -0.40
Linear regression   training MAE  8.24 | validation MAE  8.83 | gap +0.59
Ridge (alpha=1)     training MAE  8.23 | validation MAE  8.82 | gap +0.59
Lasso (alpha=0.01)  training MAE  8.24 | validation MAE  8.83 | gap +0.59


In [ ]:
# Choose on validation with a rule fixed in advance: lowest MAE, but prefer the simplest model within 0.1 marks
simplicity = ["Linear regression", "Ridge (alpha=1)", "Lasso (alpha=0.01)"]
val_mae = comparison.set_index("Model")["MAE"]
best = val_mae[simplicity].min()
chosen = next(n for n in simplicity if val_mae[n] <= best + 0.1)
print(f"Lowest validation MAE {best:.2f}; chosen: {chosen}")

# Then, once, the test set: the chosen model and the baseline on the same test students
for name in ["Mean baseline", chosen]:
    t = scores(test["ExamScore"], models[name].predict(test[features]))
    print(f"{name:<19} test MAE {t['MAE']:.2f} | RMSE {t['RMSE']:.2f} | R2 {t['R2']:.3f}")
pd.DataFrame({"ActualScore": y_val, "PredictedScore": models[chosen].predict(validation[features]).round(1)}).to_csv(
    "comparison_predictions.csv", index=False)

Lowest validation MAE 8.82; chosen: Linear regression
Mean baseline       test MAE 10.87 | RMSE 13.92 | R2 -0.002
Linear regression   test MAE 8.62 | RMSE 9.92 | R2 0.491


## 4. Overfitting and underfitting in regression

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

# Section 4: make the model more complex step by step and watch training vs validation error
def rmse(actual, predicted):
    return np.sqrt(mean_squared_error(actual, predicted))

rows = []
for degree in [1, 2, 3, 5, 10]:
    m = make_pipeline(PolynomialFeatures(degree, include_bias=False), StandardScaler(), LinearRegression())
    m.fit(train[features], train["ExamScore"])
    rows.append({"Degree": f"Degree {degree}", "TrainRMSE": rmse(train["ExamScore"], m.predict(train[features])),
                 "ValidationRMSE": rmse(y_val, m.predict(validation[features])), "ValidationR2": r2_score(y_val, m.predict(validation[features]))})
complexity = pd.DataFrame(rows).round(3)
print(complexity.to_string(index=False))
complexity.to_csv("complexity.csv", index=False)

   Degree  TrainRMSE  ValidationRMSE  ValidationR2
 Degree 1     10.550          10.500         0.312
 Degree 2     10.272          11.224         0.214
 Degree 3      9.609          11.400         0.190
 Degree 5      8.359          32.531        -5.600
Degree 10      7.684          41.349        -9.663


In [ ]:
# The same experiment with study hours alone: only 14 different values, so extra degrees have little to bend around
print("Different study-hour values in training:", train["StudyHours"].nunique())
for degree in [1, 2, 3, 5, 10]:
    m = make_pipeline(PolynomialFeatures(degree, include_bias=False), StandardScaler(), LinearRegression())
    m.fit(train[["StudyHours"]], train["ExamScore"])
    print(f"Degree {degree:>2}: training RMSE {rmse(train['ExamScore'], m.predict(train[['StudyHours']])):5.2f}"
          f" | validation RMSE {rmse(y_val, m.predict(validation[['StudyHours']])):5.2f}")

Different study-hour values in training: 14
Degree  1: training RMSE 11.15 | validation RMSE 11.07
Degree  2: training RMSE 11.13 | validation RMSE 11.02
Degree  3: training RMSE 11.11 | validation RMSE 10.98


Degree  5: training RMSE 10.82 | validation RMSE 11.27
Degree 10: training RMSE 10.74 | validation RMSE 11.43


In [ ]:
from sklearn.tree import DecisionTreeRegressor

# Both failures in one family: a decision tree from very shallow to unlimited depth
rows = []
for depth in [1, 2, 3, 5, None]:
    m = DecisionTreeRegressor(max_depth=depth, random_state=42).fit(train[features], train["ExamScore"])
    rows.append({"Depth": "Unlimited" if depth is None else f"Depth {depth}",
                 "TrainRMSE": rmse(train["ExamScore"], m.predict(train[features])),
                 "ValidationRMSE": rmse(y_val, m.predict(validation[features]))})
trees = pd.DataFrame(rows).round(2)
print(trees.to_string(index=False))
trees.to_csv("tree_depths.csv", index=False)

    Depth  TrainRMSE  ValidationRMSE
  Depth 1      12.08           12.53
  Depth 2      10.82           12.07
  Depth 3       9.56           13.26
  Depth 5       6.80           13.74
Unlimited       2.04           15.51


In [ ]:
# Reducing overfitting: keep the degree-5 features but add a Ridge penalty, chosen on validation
for alpha in [0.01, 1, 10, 100]:
    m = make_pipeline(PolynomialFeatures(5, include_bias=False), StandardScaler(), Ridge(alpha=alpha))
    m.fit(train[features], train["ExamScore"])
    print(f"Degree 5 + Ridge alpha {alpha:>6}: training RMSE {rmse(train['ExamScore'], m.predict(train[features])):5.2f}"
          f" | validation RMSE {rmse(y_val, m.predict(validation[features])):5.2f}")
print(f"Plain linear regression (degree 1):  validation RMSE {rmse(y_val, models['Linear regression'].predict(validation[features])):5.2f}")

Degree 5 + Ridge alpha   0.01: training RMSE  9.50 | validation RMSE 10.95
Degree 5 + Ridge alpha      1: training RMSE  9.99 | validation RMSE 10.69


Degree 5 + Ridge alpha     10: training RMSE 10.30 | validation RMSE 10.43
Degree 5 + Ridge alpha    100: training RMSE 10.67 | validation RMSE 10.25
Plain linear regression (degree 1):  validation RMSE 10.50


## 5. Cross-validation for regression

In [ ]:
# Section 5: why one split isn't enough. Same model, same 127 students, different 23-student validation sets
single = []
for seed in range(10):
    fit_part, check_part = train_test_split(train_val, test_size=23, random_state=seed)
    p = LinearRegression().fit(fit_part[features], fit_part["ExamScore"]).predict(check_part[features])
    single.append(rmse(check_part["ExamScore"], p))
print("Validation RMSE for 10 random splits:", np.round(single, 2).tolist())
print(f"From {min(single):.2f} to {max(single):.2f}, depending only on which students were held out")

Validation RMSE for 10 random splits: [12.97, 8.67, 9.26, 10.39, 10.55, 10.59, 10.01, 11.31, 11.69, 10.4]
From 8.67 to 12.97, depending only on which students were held out


In [ ]:
from sklearn.model_selection import KFold, cross_validate

# 5-fold cross-validation on the 127 non-test students: every student is in a validation fold exactly once.
# Scaling sits inside each pipeline, so it is refitted on each fold's training part (no leakage).
X_cv, y_cv = train_val[features], train_val["ExamScore"]
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_models = {
    "Linear": LinearRegression(),
    "Ridge": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    "Lasso": make_pipeline(StandardScaler(), Lasso(alpha=0.01)),
}
rows, folds = [], []
for name, m in cv_models.items():
    cv = cross_validate(m, X_cv, y_cv, cv=kf, scoring={"rmse": "neg_root_mean_squared_error", "r2": "r2"})
    fold_rmse = -cv["test_rmse"]
    rows.append({"Model": name, "RMSE_mean": fold_rmse.mean(), "RMSE_std": fold_rmse.std(),
                 "R2_mean": cv["test_r2"].mean(), "R2_std": cv["test_r2"].std()})
    if name == "Linear":
        print("Linear regression, RMSE per fold:", np.round(fold_rmse, 2).tolist())
        folds = [{"Fold": f"Fold {i + 1}", "RMSE": round(v, 2)} for i, v in enumerate(fold_rmse)]
cv_results = pd.DataFrame(rows).round(3)
print(cv_results.to_string(index=False))
cv_results.to_csv("cv_results.csv", index=False)
pd.DataFrame(folds).to_csv("cv_folds.csv", index=False)

Linear regression, RMSE per fold: [10.33, 12.99, 10.58, 8.83, 11.65]
 Model  RMSE_mean  RMSE_std  R2_mean  R2_std
Linear     10.875     1.388    0.309   0.154
 Ridge     10.870     1.401    0.310   0.152
 Lasso     10.875     1.391    0.309   0.154


In [ ]:
from sklearn.model_selection import RepeatedKFold

# How many folds? More folds, smaller validation parts, noisier individual scores; repeats average out the luck
for label, splitter in [("5-fold", KFold(5, shuffle=True, random_state=42)),
                        ("10-fold", KFold(10, shuffle=True, random_state=42)),
                        ("5-fold repeated 10 times", RepeatedKFold(n_splits=5, n_repeats=10, random_state=42))]:
    scores = -cross_validate(LinearRegression(), X_cv, y_cv, cv=splitter, scoring="neg_root_mean_squared_error")["test_score"]
    print(f"{label:<25} {len(scores):>2} scores: RMSE {scores.mean():.2f} +/- {scores.std():.2f}")

5-fold                     5 scores: RMSE 10.88 +/- 1.39
10-fold                   10 scores: RMSE 10.58 +/- 2.48
5-fold repeated 10 times  50 scores: RMSE 10.86 +/- 1.11


In [ ]:
from sklearn.model_selection import GridSearchCV

# Choose with cross-validation, then test once: polynomial degree and Ridge alpha, judged by 5-fold CV RMSE
grid = GridSearchCV(
    make_pipeline(PolynomialFeatures(include_bias=False), StandardScaler(), Ridge()),
    {"polynomialfeatures__degree": [1, 2, 3], "ridge__alpha": [0.1, 1, 10, 100]},
    cv=kf, scoring="neg_root_mean_squared_error",
).fit(X_cv, y_cv)
best = grid.best_params_
print(f"Chosen by CV: degree {best['polynomialfeatures__degree']}, alpha {best['ridge__alpha']}, CV RMSE {-grid.best_score_:.2f}")
print(f"Test RMSE, once: {rmse(test['ExamScore'], grid.predict(test[features])):.2f}")

Chosen by CV: degree 1, alpha 10, CV RMSE 10.84
Test RMSE, once: 9.90


## 6. Mini project: regression evaluation

In [ ]:
# Mini project. 1: explore the data
df = pd.read_csv("student_data_150.csv")
features = ["StudyHours", "Attendance", "Age"]
print("Rows:", len(df), "| missing values in features and target:", int(df[features + ["ExamScore"]].isna().sum().sum()))
print(df[features + ["ExamScore"]].describe().loc[["mean", "std", "min", "max"]].round(1).to_string())
print("Correlation with ExamScore:", df[features].corrwith(df["ExamScore"]).round(2).to_dict())

Rows: 150 | missing values in features and target: 0
      StudyHours  Attendance   Age  ExamScore
mean         3.6        79.5  11.0       57.4
std          1.4        10.0   0.8       13.9
min          0.5        54.0  10.0       24.0
max          7.0       100.0  12.0       91.0
Correlation with ExamScore: {'StudyHours': 0.61, 'Attendance': 0.38, 'Age': 0.08}


In [ ]:
# 2-5: keep a test set aside, then compare four models with 5-fold cross-validation on the rest, all four metrics
train_val, test = train_test_split(df, test_size=0.15, random_state=42)
X_cv, y_cv = train_val[features], train_val["ExamScore"]
kf = KFold(n_splits=5, shuffle=True, random_state=42)
project_models = {
    "Mean baseline": DummyRegressor(strategy="mean"),
    "Linear": LinearRegression(),
    "Ridge": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    "Lasso": make_pipeline(StandardScaler(), Lasso(alpha=0.01)),
}
scoring = {"mae": "neg_mean_absolute_error", "mse": "neg_mean_squared_error", "rmse": "neg_root_mean_squared_error", "r2": "r2"}
rows = []
for name, m in project_models.items():
    cv = cross_validate(m, X_cv, y_cv, cv=kf, scoring=scoring)
    rows.append({"Model": name, "MAE": -cv["test_mae"].mean(), "MSE": -cv["test_mse"].mean(),
                 "RMSE": -cv["test_rmse"].mean(), "RMSE_std": cv["test_rmse"].std(), "R2": cv["test_r2"].mean()})
project_cv = pd.DataFrame(rows).round(3)
print(f"5-fold cross-validation on {len(train_val)} students (test set of {len(test)} kept aside)")
print(project_cv.to_string(index=False))
project_cv.to_csv("project_cv.csv", index=False)

5-fold cross-validation on 127 students (test set of 23 kept aside)
        Model    MAE     MSE   RMSE  RMSE_std     R2
Mean baseline 10.959 191.815 13.593     2.655 -0.036
       Linear  8.603 120.201 10.875     1.388  0.309
        Ridge  8.595 120.111 10.870     1.401  0.310
        Lasso  8.603 120.208 10.875     1.391  0.309


In [ ]:
# 6: choose by a rule fixed in advance (lowest CV RMSE; the simplest model within 0.1 marks), then test ONCE
cv_rmse = project_cv.set_index("Model")["RMSE"]
candidates = ["Linear", "Ridge", "Lasso"]                     # simplest first
chosen = next(n for n in candidates if cv_rmse[n] <= cv_rmse[candidates].min() + 0.1)
print("Chosen on cross-validation:", chosen)
final = project_models[chosen].fit(X_cv, y_cv)                # refit on all 127 non-test students
baseline = project_models["Mean baseline"].fit(X_cv, y_cv)
for name, m in [("Mean baseline", baseline), (chosen, final)]:
    p = m.predict(test[features])
    print(f"{name:<14} test MAE {mean_absolute_error(test['ExamScore'], p):5.2f} | MSE {mean_squared_error(test['ExamScore'], p):6.2f}"
          f" | RMSE {rmse(test['ExamScore'], p):5.2f} | R2 {r2_score(test['ExamScore'], p):.3f}")
pd.DataFrame({"ActualScore": test["ExamScore"], "PredictedScore": final.predict(test[features]).round(1)}).to_csv(
    "project_test_predictions.csv", index=False)

Chosen on cross-validation: Linear
Mean baseline  test MAE 10.86 | MSE 193.65 | RMSE 13.92 | R2 -0.001
Linear         test MAE  8.68 | MSE  97.66 | RMSE  9.88 | R2 0.495


In [ ]:
# Findings: what the chosen model learned, and where it is weakest
print(f"Intercept {final.intercept_:.2f} |", " | ".join(f"{c} {b:+.2f}" for c, b in zip(features, final.coef_)))
errors = test.assign(Predicted=final.predict(test[features]).round(1))
errors["Error"] = (errors["ExamScore"] - errors["Predicted"]).round(1)
print(f"Test errors within 10 marks: {(errors['Error'].abs() <= 10).mean():.0%} of {len(test)} students")
print("Largest test errors:")
print(errors.reindex(errors["Error"].abs().sort_values(ascending=False).index)
      [["Student", "StudyHours", "Attendance", "ExamScore", "Predicted", "Error"]].head(3).to_string(index=False))

Intercept 0.92 | StudyHours +5.47 | Attendance +0.37 | Age +0.70
Test errors within 10 marks: 70% of 23 students
Largest test errors:
Student  StudyHours  Attendance  ExamScore  Predicted  Error
   G062         1.0        74.0       24.0       41.3  -17.3
   G024         1.5        78.0       62.0       45.5   16.5
   G097         3.5        77.0       73.0       56.8   16.2
